In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
import os
from zoneinfo import ZoneInfo
from typing import Dict, Iterable, Union, Tuple, List, Literal, Mapping, Optional, Any

from frequenz.data.microgrid import component_data
from frequenz.data.microgrid.config import MicrogridConfig
import frequenz.lib.notebooks.reporting.plotter as pl
from frequenz.lib.notebooks.reporting.data_processing import *
from frequenz.lib.notebooks.reporting.utils.helpers import *
from frequenz.client.common.metric import Metric        
from frequenz.client.reporting import ReportingApiClient 
from IPython.display import display, Markdown

import logging
logging.basicConfig()
logging.getLogger("frequenz.lib.notebooks").setLevel(logging.WARNING)

pv_production_sum=0
net_site_consumption_sum=0
grid_consumption_sum=0
pv_feed_in_sum=0
peak=0
pv_self_consumption_sum=0
pv_self_consumption_share=0
pv_total_consumption_share=0

# Reporting Notebook

⚠️ **Wichtig:**
Um die Anweisungen anzuzeigen (falls nicht bereits angezeigt), müssen Sie **die nächste Zelle ausführen**.

▶️ **Wie führe ich die nächste Zelle aus?**
--> Drücken Sie die **Run** ▶️-Taste oben.


In [4]:
%%html
<style>
  details {
    background: #f8f9fa;
    border-radius: 6px;
    padding: 12px;
    margin-bottom: 12px;
    border: 1px solid #ddd;
  }

  summary {
    font-size: 18px;
    font-weight: bold;
    cursor: pointer;
    padding: 6px;
    border-radius: 4px;
  }

  summary:hover {
    color: #007bff;
  }

  p {
    font-size: 15px;
    line-height: 1.6;
  }
  
  ul {
    margin-left: 20px;
    padding-left: 15px;
    font-size: 15px;
  }
  
  li {
    margin-bottom: 8px;
  }
  
  /* Style checked items */
  .checked {
    text-decoration: line-through;
    color: gray;
  }
</style>

<details>
  <summary>🇩🇪 <strong>Deutsche Anleitung (Klicken zum Anzeigen)</strong></summary>

  <p><strong>🛠️ Anleitung</strong></p>
  <p>Das Notebook benötigt einige Einstellungen, bevor es zum ersten Mal ausgeführt werden kann.</p>

  <ul>
    <li><input type="checkbox" onclick="toggleStrike(this)"> ✅ <strong>Neueste Paketversion sicherstellen:</strong> Es wird empfohlen, die neuesten Version der benötigten Pakete zu installieren. Alle Versionen finden Sie hier: <a href="https://pypi.org/project/frequenz-lib-notebooks/#history">frequenz-lib-notebooks</a>, <a href="https://pypi.org/project/frequenz-client-reporting/#history">frequenz-client-reporting</a>. Ersetzen Sie bei der Installation die vordefinierte Version durch die gewünschte.</li>
    <li><input type="checkbox" onclick="toggleStrike(this)"> 🛠️ <strong>Führen Sie die erste Code-Zelle aus:</strong> Klicken Sie auf das <strong><span style="color: blue;">blaue</span> Dreieck</strong>, um Pakete zu installieren. Danach verschieben Sie sie zu <strong>"requirements.txt"</strong> (weitere Informationen finden Sie im Dokument <em>"Anleitung: Deepnote Projekt erstellen und Notebook importieren"</em>, Abschnitt <em>"5. Requirements installieren"</em>).</li>
    <li><input type="checkbox" onclick="toggleStrike(this)"> 📦 <strong>Importieren Sie die Bibliotheken:</strong> Führen Sie die nächsten zwei Code-Zellen aus, um die notwendigen Bibliotheken zu importieren und die Datei <code>microgrids.toml</code> zu laden. ⏳ <em>Warten Sie, bis die Ausführung abgeschlossen ist.</em></li>
    <li><input type="checkbox" onclick="toggleStrike(this)"> 🎯 <strong>Konfigurieren Sie die Eingaben:</strong> Füllen Sie die erforderlichen Felder unter <strong>"Input"</strong> aus. </li>
    <li><input type="checkbox" onclick="toggleStrike(this)"> 🚀 <strong>Notebook ausführen:</strong> Klicken Sie auf den <strong><span style="color: blue;">"Start"</span>-Button</strong>, sobald alle Eingaben ausgefüllt sind.</li>
  </ul>

</details>

<script>
  function toggleStrike(checkbox) {
    if (checkbox.checked) {
      checkbox.parentElement.classList.add("checked");
    } else {
      checkbox.parentElement.classList.remove("checked");
    }
  }
</script>

# Intro

In diesem Notebook können Sie die Daten aus der Reporting API analysieren und visualisieren. Sie haben die Möglichkeit, den gewünschten Zeitraum und die Auflösung der zu analysierenden Daten festzulegen.
Die Analyse umfasst wichtige Kennzahlen zur Stromgewinnung aus Photovoltaikanlagen (PV), zur Nutzung von Batteriespeichern sowie zum Netzanschluss. Die Ergebnisse werden in anschaulichen Charts dargestellt, um Ihnen einen klaren Überblick über die Energieflüsse zu ermöglichen.
Wählen Sie einfach Ihre gewünschten Parameter aus, um tiefere Einblicke in Ihre Energieerzeugung und -nutzung zu erhalten.

# Inputs

In [5]:
directory = "/work"
toml_files = [f for f in os.listdir(directory) if f.endswith(".toml")]
if not toml_files:
    raise FileNotFoundError("No .toml files found in /work.")

configs: dict[str, "MicrogridConfig"] = {}
for toml_file in toml_files:
    configs.update(MicrogridConfig.load_configs(toml_file))
available_microgrids = sorted(list(int(x) for x in configs.keys()))

Um das Notebook nutzen zu können müssen zunächst die API Anmeldeinformationen hinterlegt werden (Anleitung). Die API Anmeldeinformationen finden Sie im Kuiper-Portal. Dort finden Sie ebenfalls ihre Microgrid ID, die Meter PQ ID, die Meter Batterie IDs sowie die Meter PV IDs. Bitte geben Sie Ihre Komponenten IDs unten ein, wählen sie ein Start- und Enddatum, wählen sie eine Auflösung und klicken Sie auf Start, um das Notebook auszuführen.
Eine Anleitung zum Vorgehen finden Sie hier.

In [6]:
microgrid_id = '13'

Zeitrahmen

In [7]:

from dateutil.parser import parse as _deepnote_parse
start_date = _deepnote_parse('2025-08-01T00:00:00.000Z').date()


In [8]:

from dateutil.parser import parse as _deepnote_parse
end_date = _deepnote_parse('2025-08-31T00:00:00.000Z').date()


In [9]:
resolution = '15'

run_notebook = False

In [11]:
# Casting input values
def cast_input(input_string, output_type=int):
    if input_string == 'microgrid_id':
        try :
            globals()[f'{input_string}'] = int(globals()[f'{input_string}'])
            # Check if valid microgrid id
        except:
            globals()[f'{input_string}'] = 0
            Exception(input_string + ' muss ganzzahlig sein')
    if input_string == 'resolution':
        try:
            current_val = globals()[input_string]

            # Only convert if it's not already a timedelta
            if not isinstance(current_val, timedelta):
                globals()[input_string] = timedelta(seconds=int(current_val) * 60)

        except Exception:
            globals()[input_string] = timedelta(seconds=0)
            raise Exception(input_string + ' muss ganzzahlig sein')
    elif output_type == int:
        try:
            globals()[f'{input_string}'] = int(globals()[f'{input_string}'])
        except:
            globals()[f'{input_string}'] = 0
            raise Exception(input_string + ' muss ganzzahlig sein')

    elif output_type == list:  # Added list
        try:
            strings = globals()[f'{input_string}'].split(',')
            globals()[f'{input_string}'] = [int(x) for x in strings]
        except Exception as e:
            raise Exception(f'Prüfen Sie die {input_string}: {str(e)}')

    elif output_type == int:
        try:
            globals()[f'{input_string}'] = int(globals()[f'{input_string}'])
        except:
            globals()[f'{input_string}'] = 0
            raise Exception(input_string + ' muss ganzzahlig sein')

    elif output_type == date:
        try:
            dt = globals()[f'{input_string}']
            if input_string == 'start_date':
                dt = datetime(dt.year, dt.month, dt.day, 0, 0, 0, tzinfo=ZoneInfo("CET"))
            else:
                dt = datetime(dt.year, dt.month, dt.day, 0, 0, 0, tzinfo=ZoneInfo("CET"))
            globals()[f'{input_string}'] = dt.astimezone(ZoneInfo('UTC'))
        except:
            raise Exception(input_string + ' muss ein Datumsformat sein')

if run_notebook:
    mapper = ColumnMapper.from_yaml("schema_mapping.yaml")
    variable_types = {
        'start_date': date,
        'end_date': date,
        'microgrid_id': int,
        'resolution': timedelta
    }

    for input in variable_types:
        if globals()[input]:
            cast_input(input, variable_types[input])

    if start_date >= end_date:
        raise Exception('Prüfen Sie die eingegebenen Daten. Das Startdatum muss kleiner oder gleich dem Enddatum sein.')
    else:
        start_dt = start_date
        end_dt = end_date

## Datenimport

In [12]:
if run_notebook:
    toml_path = os.path.join(directory, f'microgrids_eid{configs[str(microgrid_id)]._metadata.gid}.toml')

    mdata = component_data.MicrogridData(
        server_url=os.environ["REPORTING_SERVER_URL"],
        auth_key=os.environ["REPORTING_API_KEY"],
        sign_secret=os.environ["REPORTING_API_SECRET"],
        microgrid_config_path=toml_path,
    ) 

    mids = [i.replace("iot", "") for i in mdata.microgrid_ids] 

    mcfg = mdata.microgrid_configs[str(microgrid_id)]
    ctypes = mcfg.component_types()
    component_types = ctypes

    df = await mdata.ac_active_power(
        microgrid_id=microgrid_id, 
        component_types=component_types,
        start=start_date,
        end=end_date,
        resampling_period=resolution,
        keep_components=True,
        splits=True,
    )

    print(f"Received {df.shape[0]} rows and {df.shape[1]} columns")
    # df.to_csv('raw_df.csv')

else:
    df = pd.DataFrame()

df

""


# Output

In [13]:
if run_notebook:
    df = _add_pv_energy_flows(df)
    df = df.reset_index()
    df = mapper.to_canonical(df)

    energy_report_df = create_energy_report_dfs(df, component_types, mcfg)

    # base_cols = ['Zeitpunkt', 'Netzbezug', 'Netto Gesamtverbrauch']
    base_cols = ['timestamp', 'net_import', 'net_consumption']
    optional_cols = {
        # 'pv': ['PV Produktion', 'PV Einspeisung'],
        # 'battery': ['Batterie Durchsatz'],
        'pv': ['pv_prod', 'pv_feedin'],
        'battery': ['battery_throughput'],

    }

    # Start with base columns
    cols = base_cols[:]

    # Add optional columns if the component is present
    for comp, comp_cols in optional_cols.items():
        if comp in component_types:
            cols.extend(comp_cols)

    overview_df = energy_report_df[cols]

In [14]:
if run_notebook:
    grid_consumption_sum = (energy_report_df.get("net_import", 0) * (resolution.seconds / 3600)).sum()
    energy_summary_df = compute_energy_summary(energy_report_df, resolution=resolution)
    energy_metrics_dict = aggregate_pv_metrics(energy_report_df=energy_report_df, resolution=resolution, grid_consumption_sum=grid_consumption_sum)

    (pv_feed_in_sum,
    pv_production_sum,
    pv_self_consumption_sum,
    pv_bat_sum,
    pv_self_consumption_share,
    pv_total_consumption_share,
    net_site_consumption_sum,
    peak,
    peak_date) = aggregate_pv_metrics(energy_report_df, resolution, grid_consumption_sum).values()

else:
    energy_summary_df = pd.DataFrame()

energy_summary_df

""


## Übersicht

In [15]:
if run_notebook:
    print(f'{datetime.strftime(start_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")} - {datetime.strftime(end_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")}')

In [16]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("Brutto Stromverbrauch kWh")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{net_site_consumption_sum}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "Brutto Stromverbrauch kWh", "value": "0"}'

In [17]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("Bezug in kWh")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{grid_consumption_sum}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "Bezug in kWh", "value": "0"}'

In [18]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("PV Eigenverbrauch in kWh")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{pv_self_consumption_sum}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "PV Eigenverbrauch in kWh", "value": "0"}'

In [19]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("PV Gesamterzeugung in kWh")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{pv_production_sum}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "PV Gesamterzeugung in kWh", "value": "0"}'

In [20]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("Einspeisung in kWh")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{pv_feed_in_sum}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "Einspeisung in kWh", "value": "0"}'

In [21]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("Lastspitze in kW {{peak_date}}")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{peak}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "Lastspitze in kW None", "value": "0"}'

Lastgang im zeitlichen Verlauf: 
Hier werden die PV-Gesamtleistung, der Verbrauch (Last), der Netzbezug, die (PV) Einspeisung und die Batterie-Gesamtleistung im ausgewählten Zeitraum dargestellt.

In [22]:
if run_notebook:
    overview_df = mapper.to_display(overview_df)
    fig = plot_time_series(overview_df, time_col="Zeitpunkt", cols=None, title="Lastgang Übersicht")
    fig.show()

Energiebezug
Hier sehen Sie das Verhältnis von Eigenverbrauch und Netzbezug in dem von Ihnen gewählten Zeitraum an.

In [23]:
if run_notebook:
    energy_summary_df = energy_summary_df.rename(columns={"Energy Source": "Energiebezug", "Energy [kWh]": "Energie [kWh]"})
    pl.plot_energy_pie_chart(energy_summary_df)

## PV

In [24]:
if run_notebook:
    print(f'{datetime.strftime(start_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")} - {datetime.strftime(end_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")}')

In [25]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("PV Gesamterzeugung in kWh")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{pv_production_sum}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "PV Gesamterzeugung in kWh", "value": "0"}'

In [26]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("PV Eigenverbrauch in kWh")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{pv_self_consumption_sum}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "PV Eigenverbrauch in kWh", "value": "0"}'

In [27]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("PV Eigenverbrauch in %")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{pv_self_consumption_share}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "PV Eigenverbrauch in %", "value": "0"}'

In [28]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template("PV-Anteil Gesamtvb. (%)")
    rendered_comparison_title = render_template("")

    return json.dumps({
        "comparisonTitle": rendered_comparison_title,
        "comparisonValue": "",
        "title": rendered_title,
        "value": f"{pv_total_consumption_share}"
    })

__deepnote_big_number__()


'{"comparisonTitle": "", "comparisonValue": "", "title": "PV-Anteil Gesamtvb. (%)", "value": "0"}'

In [29]:
if run_notebook:
    _ = display_pv_energy(energy_report_df, resolution)

PV-Leistung 
Hier sehen Sie sowohl die kumulierte Leistung der PV-Anlagen als auch die gestapelten Einzelleistungen der jeweiligen PV-Anlagen. Die Netzanschluss-Kurve zeigt an, wie viel Stromüberschüsse ins Netz eingespeist wurden (Kurve im negativen Bereich). Beziehungsweise wieviel Strom vom Netz bezogen wurde (Kurve im positiven Bereich).
Um einzelne PV-Anlagen zu filtern muss zunächst "Alle" aus dem Filter entfernt und dann die gewünschten PV IDs ausgewählt werden.

In [30]:
if run_notebook:
    pv_columns = [c for c in energy_report_df.columns if "PV #" in c]
    if 'pv' in component_types:
        pv_grid_filter_options = ["PV und Netzanschluss", "Nur PV", "Nur Netzanschluss"]
        pv_filter_options = ['Alle'] + [pv[3:] for pv in pv_columns]

In [31]:
pv_filter = ['Alle']

Filter_pv = False

In [33]:
if run_notebook or Filter_pv:
    pv_analyse_df = pd.DataFrame()
    pv_filter = ["All" if x == "Alle" else x for x in pv_filter]

    if 'pv' in component_types and 'timestamp' in energy_report_df.columns:
        pv_analyse_df = build_pv_analysis_df(energy_report_df, pv_filter)
        pv_analyse_df = mapper.to_display(pv_analyse_df)

else:
    pv_analyse_df = pd.DataFrame()
pv_analyse_df

""


In [34]:
if run_notebook:
    if "pv" in component_types:
        fig = plot_time_series(pv_analyse_df, time_col="Zeitpunkt", cols=["Netzanschluss", "PV Einspeisung"], title="PV Leistung")
        fig.show()

## Batterie

In [35]:
if run_notebook:
    print(f'{datetime.strftime(start_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")} - {datetime.strftime(end_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")}')

Batterieleistungskurve 
Die Batterieleistungs-Kurve zeigt kumuliert für alle Batterien an, wie viel Strom aus der Batterie bezogen wurde (Kurve im negativen Bereich). Beziehungsweise wieviel Strom in die Batterie eingespeist wurde (Kurve im positiven Bereich). 
Um einzelne Batterien zu filtern muss zunächst "Alle" aus dem Filter entfernt und dann die gewünschten Batterie IDs ausgewählt werden.

In [36]:
if run_notebook:
    if 'battery' in component_types:
        battery_filter_options = ['Alle'] + [f'#{i}' for i in mcfg.component_type_ids('battery')]

In [37]:
bat_filter = ['Alle']

filter_battery = False

In [39]:
if run_notebook or filter_battery:
    if "battery" in {str(x).lower() for x in component_types}:
        bat_filter = ["All" if x == "Alle" else x for x in bat_filter]
        bat_analyse_df = build_component_analysis(energy_report_df, bat_filter, 'Battery', 'Batterie Durchsatz')
        bat_analyse_df = mapper.to_display(bat_analyse_df)
        fig = plot_time_series(bat_analyse_df, time_col="Zeitpunkt", cols=["Batterie Durchsatz"], title="Batterie Durchsatz", legend_title=None)
        fig.show()

## Netzbezug

In [40]:
if run_notebook:
    print(f'{datetime.strftime(start_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")} - {datetime.strftime(end_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")}')

Bezugskurve
Die Bezugskurve zeigt wie viel Strom aus dem Stromnetz bezogen wurde.

In [41]:
if run_notebook:
    fig = plot_time_series(overview_df, time_col="Zeitpunkt", cols=["Netzbezug"], title="Netzbezug")
    fig.show()

# CHP

In [42]:
if run_notebook:
    if 'chp' in component_types:
        chp_filter_options = ['Alle'] + [f'#{i}' for i in mcfg.component_type_ids('chp')]

In [43]:
chp_filter = ['Alle']

filter_chp = False

In [45]:
if run_notebook or filter_chp:
    if "chp" in {str(x).lower() for x in component_types}:
        chp_filter = ["All" if x == "Alle" else x for x in chp_filter]
        chp_analyse_df = build_component_analysis(energy_report_df, chp_filter, 'CHP', 'CHP Usage')
        chp_analyse_df = mapper.to_display(chp_analyse_df)
        fig = plot_time_series(chp_analyse_df, time_col="Zeitpunkt", cols=["CHP Usage"], title="CHP", legend_title=None)
        fig.show()

# EV

In [46]:
if run_notebook:
    if 'ev' in component_types:
        ev_filter_options = ['Alle'] + [f'#{i}' for i in mcfg.component_type_ids('ev')]

In [47]:
ev_filter = ['Alle']

filter_ev = False

In [49]:
if run_notebook or filter_ev:
    if "ev" in {str(x).lower() for x in component_types}:
        ev_filter = ["All" if x == "Alle" else x for x in ev_filter]
        ev_analyse_df = build_component_analysis(energy_report_df, ev_filter, 'EV', 'EV Usage')
        ev_analyse_df = mapper.to_display(ev_analyse_df)
        fig = plot_time_series(ev_analyse_df, time_col="Zeitpunkt", cols=["EV Usage"], title="EV", legend_title=None)
        fig.show()

## Daten

In [50]:
if run_notebook:
    print(f'{datetime.strftime(start_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")} - {datetime.strftime(end_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")}')

### Daten exportieren

Die Daten werden unter angegebenem Dateinamen in den Deepnote Files gespeichert und können von dort heruntergeladen werden.
Mehr erfahren

In [51]:
output = ''

In [52]:
if output:
    if os.path.exists(output):
        print("Fehler: Die Daten wurden nicht exportiert. Es gibt bereits eine Datei mit diesem Namen.")
    elif output.endswith(".csv"):
        master_df['Zeitpunkt'] = master_df['Zeitpunkt'].dt.strftime("%d.%m.%Y %H:%M:%S")
        master_df.to_csv(output, index=False)
        print(f"Die Daten wurden erfolgreich in {output} exportiert.")
    elif output.endswith(".xlsx"):
        master_df['Zeitpunkt'] = master_df['Zeitpunkt'].dt.strftime("%d.%m.%Y %H:%M:%S")
        master_df.to_excel(output, index=False)
        print(f"Die Daten wurden erfolgreich in {output} exportiert.")
    else:
        print("Fehler: Die Daten wurden nicht exportiert. Output muss .csv oder .xlsx sein.")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=63aabf86-f060-41e1-8fed-c9781470d822' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>